# Clean & Merge DPIRD stations

We only keep weather stations with fixed position AND owned by Department of Primary Industries and Regional Development (DPIRD)

In [ ]:
import pandas as pd
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from functools import partial

## Clean for DPIRD-only station

In [ ]:
"""Check if station file is valid, delete if not, keep otherwise"""
def check_and_delete(file_path, valid_stations_set):
    station_name=file_path.stem
    if station_name not in valid_stations_set:
        print(f"Deleting non-DPIRD station: {file_path.name}")
        file_path.unlink(missing_ok=True)
        return 1
    return 0

def filter_non_DPIRD(base_dir_path,valid_station_csv):
    base_path= Path(base_dir_path)
    valid_station_csv= Path(valid_station_csv)
    valid_df= pd.read_csv(valid_station_csv)
    valid_stations= set(valid_df['name'].values)

    # Recursively find all csv using rglob
    all_files= list(base_path.rglob("*.csv"))
    print(f"Scanning {len(all_files)} total files against {len(valid_stations)} valid stations...")

    #Delete concurrently
    deleted_count = 0
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        bound_checker = partial(check_and_delete, valid_stations_set=valid_stations)
        results = executor.map(bound_checker, all_files)
        deleted_count = sum(results)
        
    print(f"Cleanup complete. Removed {deleted_count} non-DPIRD station records.")

In [ ]:
base_dir_path= "DPIRD_staging/dataset_DPIRD_utc0"
valid_station_csv= "configs/valid_station_coordinates.csv"
filter_non_DPIRD(base_dir_path,valid_station_csv)

## Merge station files